# ZMHK 2 · Bot privado en Google Colab
Ejecuta las celdas en orden con un runtime **CPU**. El código se carga directamente desde GitHub; no necesitas descargar un ZIP.

**Antes de empezar:** revoca las credenciales antiguas publicadas en Git. Añade los secretos nuevos `BOT_TOKEN` y `OWNER_ID` al panel de la llave de Colab y permite acceso a este notebook. No escribas tokens en celdas ni compartas salidas sensibles.

Esta reescritura incluye enlaces directos, yt-dlp y rclone opcional. **No incluye torrents, MongoDB, RSS ni sesiones de usuario.** Colab no es alojamiento 24/7 y puede terminar el runtime. Solo descarga contenido autorizado. Telegram limita las descargas del bot a 20 MB y esta versión limita envíos a 49 MB.


## 1 · Cargar el código
Deja **GitHub** seleccionado para descargar esta rama directamente. No necesitas subir archivos desde Arena. Si el repositorio es privado y el runtime no puede acceder, genera el paquete con `python tools/package_colab.py` y selecciona **ZIP**. No introduzcas tokens de GitHub en la URL.


In [ ]:
import io
import os
from pathlib import Path, PurePosixPath
import stat
import tempfile
import zipfile
from google.colab import files

SOURCE = "GitHub" #@param ["GitHub", "ZIP"]
if SOURCE == "GitHub":
    import subprocess
    PROJECT = Path(tempfile.mkdtemp(prefix="zmhk-", dir="/content"))
    result = subprocess.run([
        "git", "clone", "--depth", "1", "--single-branch",
        "--branch", "arena/44c5e389-zmhk-deploy", "--",
        "https://github.com/leomarmol03/ZMHK-DEPLOY.git", str(PROJECT),
    ], stdin=subprocess.DEVNULL, capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError("No se pudo descargar la rama desde GitHub. Comprueba que esté publicada y sea accesible. Para repositorios privados puedes usar la opción ZIP.")
    print("Código cargado desde GitHub sin conectar a Telegram.")
else:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Sube únicamente ZMHK-Colab.zip.")
    archive_name, payload = next(iter(uploaded.items()))
    if not archive_name.lower().endswith(".zip") or len(payload) > 25 * 1024 * 1024:
        raise ValueError("Se esperaba un ZIP de código de menos de 25 MiB.")
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        entries = archive.infolist()
        if len(entries) > 200 or sum(e.file_size for e in entries) > 25 * 1024 * 1024:
            raise ValueError("Paquete demasiado grande.")
        for entry in entries:
            name = PurePosixPath(entry.filename)
            if name.is_absolute() or ".." in name.parts or "\\" in entry.filename or stat.S_ISLNK(entry.external_attr >> 16):
                raise ValueError("Ruta no permitida en el ZIP.")
        PROJECT = Path(tempfile.mkdtemp(prefix="zmhk-", dir="/content"))
        archive.extractall(PROJECT)
    if not (PROJECT / "bot" / "__main__.py").is_file() or not (PROJECT / "requirements.txt").is_file():
        raise ValueError("No es el paquete de ZMHK 2. Genera el ZIP desde la revisión nueva.")
    del uploaded, payload
    print("Código cargado sin conectar a Telegram.")
if not (PROJECT / "bot" / "__main__.py").is_file() or not (PROJECT / "requirements.txt").is_file():
    raise ValueError("No se encontró ZMHK 2 en el código cargado.")


## 2 · Instalar dependencias
Crea un entorno virtual aislado. Puede tardar unos minutos; no requiere GPU. Los paquetes de Colab no se reemplazan.

In [ ]:
import subprocess
import sys

if sys.version_info < (3, 10):
    raise RuntimeError("Se necesita Python 3.10 o posterior.")
VENV = PROJECT / ".venv"
subprocess.run([sys.executable, "-m", "venv", str(VENV)], check=True)
PYTHON = str(VENV / "bin" / "python")
subprocess.run([PYTHON, "-m", "pip", "install", "-q", "-r", str(PROJECT / "requirements.txt")], check=True)
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg", "rclone"], check=True)
print("Dependencias instaladas.")


## 3 · Configurar secretos
Se leen del panel de Secretos. Si faltan o no están autorizados, se solicitan de forma oculta. No se guardan en `config.env`. `AUTHORIZED_USERS` es opcional: IDs separados por comas.

In [ ]:
from getpass import getpass
from google.colab import userdata

def secret(name):
    try:
        value = userdata.get(name)
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        value = ""
    return str(value).strip() if value else getpass(f"{name}: ").strip()

BOT_ENV = os.environ.copy()
BOT_ENV["BOT_TOKEN"] = secret("BOT_TOKEN")
BOT_ENV["OWNER_ID"] = secret("OWNER_ID")
BOT_ENV["AUTHORIZED_USERS"] = ""  # Opcional: "123456,789012"
BOT_ENV["DOWNLOAD_DIR"] = str(PROJECT / "downloads")
BOT_ENV["MAX_DOWNLOAD_MB"] = "45"
BOT_ENV["TASK_TIMEOUT"] = "1800"
BOT_ENV["RCLONE_DESTINATION"] = ""
BOT_ENV.pop("RCLONE_CONFIG", None)
# Validación sin imprimir el entorno ni sus credenciales.
subprocess.run([PYTHON, "-c", "from bot.config import Settings; Settings.from_env(); print('Configuración válida.')"],
               cwd=PROJECT, env=BOT_ENV, check=True)


## 4 · rclone (opcional, omite esta celda si no usarás `/mirror`)
Sube únicamente tu `rcl.conf`/`rclone.conf` generado previamente con rclone. Contiene credenciales: no lo publiques. Se almacena solo en el disco temporal del runtime. El destino tiene formato `drive:Bot`, donde `drive` es el nombre del remoto de tu configuración.

In [ ]:
rclone_upload = files.upload()
if len(rclone_upload) != 1:
    raise ValueError("Sube solo el archivo de configuración de rclone.")
rclone_payload = next(iter(rclone_upload.values()))
if len(rclone_payload) > 1024 * 1024:
    raise ValueError("Configuración demasiado grande.")
rclone_config = PROJECT / "rcl.conf"
rclone_config.write_bytes(rclone_payload)
rclone_config.chmod(0o600)
BOT_ENV["RCLONE_CONFIG"] = str(rclone_config)
BOT_ENV["RCLONE_DESTINATION"] = input("Destino rclone (p. ej. drive:Bot): ").strip()
del rclone_upload, rclone_payload
print("rclone configurado; comprueba el remoto y sus permisos al usar /mirror.")


## 5 · Ejecutar
La celda queda ejecutándose: es normal. Envía `/start` a tu bot desde la cuenta de `OWNER_ID`.

Comandos: `/leech URL`, `/ytdl URL`, `/mirror URL`, `/status`, `/cancel`. Una tarea simultánea. También puedes responder a un archivo de hasta 20 MB con `/leech` o `/mirror`.

**Para detener:** pulsa el botón de interrupción de esta celda. Si cambias secretos, detén antes el bot y repite la configuración. No inicies otra instancia con el mismo token. No cierres el runtime esperando que siga funcionando permanentemente.

In [ ]:
import signal

if "BOT_PROCESS" in globals() and BOT_PROCESS.poll() is None:
    raise RuntimeError("Ya hay una instancia en ejecución. Detén la celda anterior.")
BOT_PROCESS = subprocess.Popen([PYTHON, "-m", "bot"], cwd=PROJECT, env=BOT_ENV, start_new_session=True)
try:
    exit_code = BOT_PROCESS.wait()
    if exit_code:
        print("El bot terminó con un error. Revisa credenciales, conexión y otras instancias.")
except KeyboardInterrupt:
    print("Deteniendo el bot y cancelando transferencias...")
    os.killpg(BOT_PROCESS.pid, signal.SIGTERM)
    try:
        BOT_PROCESS.wait(timeout=30)
    except subprocess.TimeoutExpired:
        os.killpg(BOT_PROCESS.pid, signal.SIGKILL)
        BOT_PROCESS.wait()
    print("Bot detenido.")


### Problemas frecuentes
- **Conflict / otra instancia:** detén el bot antiguo, Docker, Heroku u otro notebook con el mismo token.
- **No responde:** revisa `OWNER_ID` (ID numérico del usuario, no del chat), el token nuevo y que la última celda siga ejecutándose.
- **Video no disponible:** puede haber restricciones del sitio, DRM, necesidad de sesión o componentes adicionales. No se garantiza compatibilidad con todos los sitios.
- **Archivo grande:** baja el tamaño o usa `/mirror` para un enlace directo y configura rclone. Subir `MAX_DOWNLOAD_MB` no elimina los límites de Telegram.
- **Runtime desconectado:** vuelve a ejecutar todas las celdas. No se reanudan tareas ni se conserva la configuración temporal.
